<a href="https://colab.research.google.com/github/janithcyapa/Wearable-Biomechanics/blob/main/05.%20Video%20Annotation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **🏋🏽 Wearable Biomechanics - Video Annotation**
> MARC - Multidisciplinary AI Research Centre | UOP


This notebook serves as the command interface for the **Wearable Biomechanics** project. It bridges the gap between raw Google Drive storage and Google Colab by using the **[OpenDatasetStore](https://github.com/janithcyapa/OpenDatasetStore)** utility.

The system maintains an automated, **JSON-indexed manifest**, ensuring that every subject and experiment is strictly organized with unique IDs and searchable metadata.

* **🛠️ Framework:** [OpenDatasetStore Library](https://github.com/janithcyapa/OpenDatasetStore)


## 📚 Initial Setup & Configuration


### 🔧 One-Time Setup Instructions

#### **Step 1: Get Access to the Shared Dataset Folder**
1. Ask your **admin** to share the Google Drive folder with you
2. You'll receive an email with a "Shared with you" notification
3. Open the shared folder in Google Drive
4. The folder should be named: **`Wearable-Biomechanics`**

#### **Step 2: Add Shortcut to Your Drive**
1. In Google Drive, locate the shared folder under **"Shared with me"**
2. **Right-click** on the folder named `Wearable-Biomechanics`
3. Select **"Add shortcut to Drive"** (or **"Organize → Add shortcut"**)
4. Navigate to your **`My Drive > Datasets`** folder
   - If `Datasets` folder doesn't exist, create it first
5. Click **"Add Shortcut"**

#### **Step 3: Verify the Setup**
- Go to **`My Drive > Datasets`** in your Google Drive
- You should see **`Wearable-Biomechanics`** with a shortcut icon ⤴️
- The full path will be: **`MyDrive/Datasets/Wearable-Biomechanics`**


### ⚠️ Important Notes
- 🔑 **Permissions**: You need at least **"Editor"** access to upload data
- 🔗 **Shortcuts**: Always use the shortcut in your Drive, not the original shared folder
- 📂 **Structure**: The system will auto-create `/index`, `/raw_data`, and `/processed_data` subfolders
- 🚫 **Don't delete** the shortcut or you'll lose access to shared data


### 📁 Folder Structure After Setup
```
My Drive/
└── Datasets/
    └── Wearable-Biomechanics/
        ├── index/
        │   ├── entities_subjects.json
        │   └── entries_experiments.json
        ├── raw_data/
        │   └── experiments/
        └── related_files/
        │    └── experiments/
        └── processed_data/
            └── experiments/
```


## **🛠️ Setup OpenDatasetStore**

In [ ]:
%%capture
from google.colab import drive
import pandas as pd
import os

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# Download the latest store manager
!pip install git+https://github.com/janithcyapa/OpenDatasetStore.git

from open_dataset_store import OpenDatasetStore

# @markdown ### Store Configuration
dataset_path = "/content/drive/MyDrive/Datasets/Wearable-Biomechanics" # @param {type:"string"}

store = OpenDatasetStore(
    base_dir=dataset_path,
    backend="local", # Use "local" since you are mounting Google Drive via Colab directly
    entity_id_format="sub_{num:04d}", # You can drop the `f""` string formatting here now
    entry_id_format="{type}_{num:04d}",
    raw_filename_format="{entry_id}_{entity_id}_{ts}.csv",
    processed_filename_format="{entry_id}_{tag}.parquet"
)


In [ ]:
# @title 🎬 Frame-by-Frame Synced Video Player { display-mode: "form" }
import os
import cv2
import base64
import subprocess
from IPython.display import display, HTML

target_entry_id = "experiments_0001" # @param {type:"string"}
burn_frame_numbers = True # @param {type:"boolean"}

# 1. Download the synced video from OpenDatasetStore
os.makedirs("/tmp/video_annotator", exist_ok=True)
synced_video_path = store.download_related_file(
    entry_type="experiments",
    entry_id=target_entry_id,
    label="synced_video",
    dest_folder="/tmp/video_annotator"
)

if not synced_video_path or not os.path.exists(synced_video_path):
    raise FileNotFoundError(f"❌ Could not find 'synced_video' for {target_entry_id}.")

# 2. Extract exact FPS and Total Frames via OpenCV
cap = cv2.VideoCapture(synced_video_path)
video_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()

# 3. Optionally burn exact 0-based frame index onto the video & ensure web-compatible H.264
play_path = f"/tmp/video_annotator/{target_entry_id}_playable.mp4"
if burn_frame_numbers:
    print(f"⚙️ Preparing frame-accurate overlay video for {target_entry_id} ({total_frames} frames @ {video_fps:.2f} FPS)...")
    vf_filter = (
        "drawtext=text='Frame\\: %{n}':start_number=0:"
        "x=20:y=20:fontsize=32:fontcolor=yellow:"
        "box=1:boxcolor=black@0.75:boxborderw=8"
    )
    cmd = (
        f'ffmpeg -y -threads 4 -i "{synced_video_path}" '
        f'-vf "{vf_filter}" -c:v libx264 -preset ultrafast -crf 23 '
        f'-pix_fmt yuv420p -an "{play_path}"'
    )
    subprocess.run(cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
else:
    play_path = synced_video_path

# 4. Encode video to Base64 for lag-free local browser scrubbing
with open(play_path, "rb") as f:
    video_b64 = base64.b64encode(f.read()).decode("ascii")

html_player = f"""
<div style="background:#18181b; color:#f4f4f5; padding:16px; border-radius:10px; font-family:monospace; max-width:1150px; border:1px solid #3f3f46;">
  <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:10px;">
    <span style="font-size:16px; font-weight:bold; color:#38bdf8;">🏋🏽 {target_entry_id} — Synced Video Inspector</span>
    <span id="frame_readout" style="background:#27272a; padding:6px 14px; border-radius:6px; font-size:16px; color:#facc15; font-weight:bold; border:1px solid #52525b;">
      Frame: 0 / {total_frames - 1} (0.000s)
    </span>
  </div>

  <video id=" inspect_vid" style="width:100%; max-height:600px; background:#000; border-radius:6px;" preload="auto">
    <source src="data:video/mp4;base64,{video_b64}" type="video/mp4">
  </video>

  <!-- Scrubber -->
  <div style="margin:12px 0;">
    <input type="range" id="frame_scrubber" min="0" max="{total_frames - 1}" value="0" step="1" style="width:100%; cursor:pointer;">
  </div>

  <!-- Controls -->
  <div style="display:flex; flex-wrap:wrap; gap:8px; align-items:center; justify-content:space-between;">
    <div style="display:flex; gap:6px; align-items:center;">
      <button onclick=" togglePlay()" id="play_btn" style="padding:6px 14px; background:#2563eb; color:white; border:none; border-radius:5px; cursor:pointer; font-weight:bold;">▶ Play</button>
      <button onclick="stepFrames(-10)" style="padding:6px 10px; background:#3f3f46; color:white; border:none; border-radius:5px; cursor:pointer;">⏪ -10f</button>
      <button onclick="stepFrames(-5)" style="padding:6px 10px; background:#3f3f46; color:white; border:none; border-radius:5px; cursor:pointer;">◀◀ -5f</button>
      <button onclick="stepFrames(-1)" style="padding:6px 12px; background:#52525b; color:#facc15; border:none; border-radius:5px; cursor:pointer; font-weight:bold;">◀ -1f</button>
      <button onclick="stepFrames(1)" style="padding:6px 12px; background:#52525b; color:#facc15; border:none; border-radius:5px; cursor:pointer; font-weight:bold;">+1f ▶</button>
      <button onclick="stepFrames(5)" style="padding:6px 10px; background:#3f3f46; color:white; border:none; border-radius:5px; cursor:pointer;">+5f ▶▶</button>
      <button onclick="stepFrames(10)" style="padding:6px 10px; background:#3f3f46; color:white; border:none; border-radius:5px; cursor:pointer;">+10f ⏩</button>
    </div>

    <div style="display:flex; gap:8px; align-items:center;">
      <label>Speed:</label>
      <select id="speed_select" onchange="changeSpeed(this.value)" style="background:#27272a; color:white; padding:5px 8px; border-radius:5px; border:1px solid #52525b;">
        <option value="0.1">0.1x</option>
        <option value="0.25">0.25x</option>
        <option value="0.5">0.5x</option>
        <option value="0.75">0.75x</option>
        <option value="1.0" selected>1.0x</option>
        <option value="1.5">1.5x</option>
        <option value="2.0">2.0x</option>
      </select>

      <input type="number" id="jump_input" placeholder="Frame #" min="0" max="{total_frames - 1}" style="width:90px; padding:5px 8px; background:#27272a; color:white; border:1px solid #52525b; border-radius:5px;">
      <button onclick="jumpToInput()" style="padding:6px 10px; background:#059669; color:white; border:none; border-radius:5px; cursor:pointer;">Go</button>
    </div>
  </div>
  <div style="margin-top:8px; font-size:12px; color:#a1a1aa;">
    💡 <b>Shortcuts (click player first):</b> <kbd>Space</kbd> Play/Pause | <kbd>←</kbd>/<kbd>→</kbd> -1/+1 Frame | <kbd>Shift + ←/→</kbd> -10/+10 Frames
  </div>
</div>

<script>
  const vid = document.getElementById(" inspect_vid");
  const scrubber = document.getElementById("frame_scrubber");
  const readout = document.getElementById("frame_readout");
  const playBtn = document.getElementById("play_btn");
  const fps = {video_fps};
  const totalFrames = {total_frames};

  window.currentVideoFrame = 0;

  function getExactFrame() {{
    return Math.min(totalFrames - 1, Math.max(0, Math.round(vid.currentTime * fps)));
  }}

  function updateUI() {{
    const f = getExactFrame();
    window.currentVideoFrame = f;
    scrubber.value = f;
    readout.innerText = `Frame: ${{f}} / ${{totalFrames - 1}} (${{vid.currentTime.toFixed(3)}}s)`;
  }}

  function seekToFrame(targetFrame) {{
    vid.pause();
    playBtn.innerText = "▶ Play";
    const clamped = Math.min(totalFrames - 1, Math.max(0, targetFrame));
    // Add half-frame offset so floating point rounding lands squarely inside the frame interval
    vid.currentTime = (clamped + 0.25) / fps;
    window.currentVideoFrame = clamped;
    scrubber.value = clamped;
    readout.innerText = `Frame: ${{clamped}} / ${{totalFrames - 1}} (${{((clamped)/fps).toFixed(3)}}s)`;
  }}

  function stepFrames(delta) {{
    seekToFrame(getExactFrame() + delta);
  }}

  function togglePlay() {{
    if (vid.paused) {{
      vid.play();
      playBtn.innerText = "⏸ Pause";
    }} else {{
      vid.pause();
      playBtn.innerText = "▶ Play";
      seekToFrame(getExactFrame());
    }}
  }}

  function changeSpeed(val) {{
    vid.playbackRate = parseFloat(val);
  }}

  function jumpToInput() {{
    const val = parseInt(document.getElementById("jump_input").value);
    if (!isNaN(val)) seekToFrame(val);
  }}

  scrubber.addEventListener("input", (e) => {{
    seekToFrame(parseInt(e.target.value));
  }});

  vid.addEventListener("timeupdate", updateUI);
  vid.addEventListener("ended", () => {{ playBtn.innerText = "▶ Play"; }});

  // Keyboard navigation
  document.addEventListener("keydown", (e) => {{
    if (["INPUT", "TEXTAREA", "SELECT"].includes(document.activeElement.tagName)) return;
    if (e.code === "Space") {{
      e.preventDefault();
      togglePlay();
    }} else if (e.code === "ArrowRight") {{
      e.preventDefault();
      stepFrames(e.shiftKey ? 10 : 1);
    }} else if (e.code === "ArrowLeft") {{
      e.preventDefault();
      stepFrames(e.shiftKey ? -10 : -1);
    }}
  }});
</script>
"""

display(HTML(html_player))

In [ ]:
# @title 📝 Dynamic Multi-Rep Phase Annotator (Inputs Only) { display-mode: "form" }
import json
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

processed_tag = "rep_segments"

PHASES = [
    "start_rest",
    "setup",
    "first_pull",
    "second_pull",
    "clean",
    "middle_rest",
    "jerk_phase_one",
    "jerk_phase_two",
    "rest_up",
    "bring_down",
    "end_rest",
    "end_rep",
]

def new_rep_template():
    return {phase: 0 for phase in PHASES}

# 1. Load existing processed data from OpenDatasetStore if available
reps_annotations = None
try:
    existing_df = store.get_entry_processed_data(
        entry_type="experiments",
        entry_id=target_entry_id,
        tag=processed_tag
    )
    if existing_df is not None and not existing_df.empty:
        cols_to_load = [c for c in PHASES if c in existing_df.columns]
        reps_annotations = existing_df[cols_to_load].to_dict(orient="records")
        print(f"📂 Loaded {len(reps_annotations)} existing annotated reps from processed data ('{processed_tag}').")
except Exception:
    pass

if not reps_annotations:
    reps_annotations = [new_rep_template()]

state = {"current_rep_idx": 0}

# --- UI Controls ---
rep_selector = widgets.Dropdown(
    options=[(f"Rep {i+1}", i) for i in range(len(reps_annotations))],
    value=0,
    description="Active Rep:",
    layout=widgets.Layout(width="200px")
)

btn_add_rep = widgets.Button(description="➕ Add Rep", button_style="success", layout=widgets.Layout(width="120px"))
btn_del_rep = widgets.Button(description="🗑️ Delete Rep", button_style="danger", layout=widgets.Layout(width="130px"))
btn_save_store = widgets.Button(description="💾 Save Processed Data", button_style="primary", layout=widgets.Layout(width="200px"))
comment_input = widgets.Text(value="", placeholder="Optional comment", description="Comment:", layout=widgets.Layout(width="320px"))
status_label = widgets.HTML(value="<i>Ready to annotate.</i>")

phase_inputs = {}
phase_rows = []

for phase in PHASES:
    int_box = widgets.IntText(
        value=reps_annotations[0].get(phase, 0),
        description=phase,
        style={'description_width': '140px'},
        layout=widgets.Layout(width="340px")
    )

    def make_on_value_change(p):
        def _on_change(change):
            reps_annotations[state["current_rep_idx"]][p] = int(change["new"])
            refresh_json_preview()
        return _on_change

    int_box.observe(make_on_value_change(phase), names="value")
    phase_inputs[phase] = int_box
    phase_rows.append(int_box)

json_preview = widgets.Textarea(
    value=json.dumps(reps_annotations, indent=2),
    description="",
    layout=widgets.Layout(width="460px", height="450px")
)

def refresh_json_preview():
    json_preview.value = json.dumps(reps_annotations, indent=2)

def load_rep_into_inputs(rep_idx):
    state["current_rep_idx"] = rep_idx
    rep_data = reps_annotations[rep_idx]
    for p in PHASES:
        phase_inputs[p].value = int(rep_data.get(p, 0))
    refresh_json_preview()

def on_rep_selected(change):
    if change["new"] is not None:
        load_rep_into_inputs(change["new"])

rep_selector.observe(on_rep_selected, names="value")

def on_add_rep(_):
    reps_annotations.append(new_rep_template())
    new_idx = len(reps_annotations) - 1
    rep_selector.options = [(f"Rep {i+1}", i) for i in range(len(reps_annotations))]
    rep_selector.value = new_idx
    status_label.value = f"<span style='color:#38bdf8;'>➕ Added Rep {new_idx+1} (Total: {len(reps_annotations)} reps)</span>"
    refresh_json_preview()

def on_del_rep(_):
    if len(reps_annotations) <= 1:
        status_label.value = "<span style='color:#fbbf24;'>⚠️ At least 1 rep must remain.</span>"
        return
    removed_idx = state["current_rep_idx"]
    reps_annotations.pop(removed_idx)
    new_idx = max(0, removed_idx - 1)
    rep_selector.options = [(f"Rep {i+1}", i) for i in range(len(reps_annotations))]
    rep_selector.value = new_idx
    load_rep_into_inputs(new_idx)
    status_label.value = f"<span style='color:#f87171;'>🗑️ Removed Rep {removed_idx+1}.</span>"

def on_save_all(_):
    try:
        parsed = json.loads(json_preview.value)
        if isinstance(parsed, list):
            reps_annotations.clear()
            reps_annotations.extend(parsed)
    except Exception as e:
        status_label.value = f"<span style='color:#f87171;'>❌ Invalid JSON in preview: {e}</span>"
        return

    # Build DataFrame (1 row per rep)
    df_segments = pd.DataFrame(reps_annotations)
    df_segments.insert(0, "rep", range(1, len(df_segments) + 1))

    try:
        store.add_processed_data(
            entry_type="experiments",
            entry_id=target_entry_id,
            df=df_segments,
            tag=processed_tag,
            comment=comment_input.value
        )
        status_label.value = (
            f"<span style='color:#4ade80;'>✅ Saved {len(df_segments)} reps to processed data "
            f"(tag: <code>{processed_tag}</code>) for <b>{target_entry_id}</b>!</span>"
        )
    except Exception as e:
        status_label.value = f"<span style='color:#f87171;'>❌ Store error: {e}</span>"

btn_add_rep.on_click(on_add_rep)
btn_del_rep.on_click(on_del_rep)
btn_save_store.on_click(on_save_all)

# Layout assembly
top_bar = widgets.HBox([rep_selector, btn_add_rep, btn_del_rep, comment_input, btn_save_store])
left_panel = widgets.VBox([widgets.HTML("<b>Phase Frame Indices (0-based):</b>")] + phase_rows)
right_panel = widgets.VBox([widgets.HTML("<b>Live JSON Output (Editable):</b>"), json_preview])
main_ui = widgets.VBox([top_bar, status_label, widgets.HBox([left_panel, right_panel], layout=widgets.Layout(gap="24px"))])

display(main_ui)